In [ ]:
# # Install dependencies for Unsloth + GPT-OSS
# !pip install --upgrade -qqq uv
# !uv pip install -qqq \
#     "torch>=2.8.0" "triton>=3.4.0" numpy pillow torchvision bitsandbytes "transformers==4.56.2" \
#     "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
#     "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
#     git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
# !uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

# # Install openai_harmony (Harmony protocol tools)
# !pip install -q openai-harmony jupyter_client pandas datasets


In [ ]:
class CONFIG:
    TRAIN_SIZE = 10_000
    BATCH_SIZE = 16 
    EPOCHS = 1 
    LEARNING_RATE = 2e-4
    MAX_SEQ_LENGTH = 2048
    MODEL_PATH = None 
    KAGGLE=True

cfg = CONFIG()
cfg.MODEL_PATH = "/kaggle/input/gpt-oss-20b-model/gpt-oss-20b" if cfg.KAGGLE else "unsloth/gpt-oss-20b"

In [ ]:

if cfg.KAGGLE:
    !uv pip install --system --no-index --find-links='/kaggle/input/unsloth-library/unsloth' 'unsloth'


In [ ]:
local_files_only = True if cfg.KAGGLE else False

In [ ]:
from unsloth import FastLanguageModel
import torch
dtype = None

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = cfg.MODEL_PATH ,
    dtype = dtype, # None for auto detection
    max_seq_length = cfg.MAX_SEQ_LENGTH, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    local_files_only = local_files_only
)

In [ ]:
import pandas as pd
if not cfg.KAGGLE:
    import kagglehub
    from kagglehub import KaggleDatasetAdapter

    # Set the path to the file you'd like to load
    file_path = "filtered_low_pass_1_or_2.jsonl"

    # Load the latest version
    df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "barnobarno/nemotron-low-reasoning-pass-rate-1-2",
    file_path,
    # Provide any additional arguments like 
    # sql_query or pandas_kwargs. See the 
    # documenation for more information:
    # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
    )
if cfg.KAGGLE:
    df = pd.read_json("/kaggle/input/nemotron-low-reasoning-pass-rate-1-2/filtered_low_pass_1_or_2.jsonl", lines=True)



In [ ]:
from datasets import Dataset

train_data = df.copy()
train_data.drop(columns=["uuid","original_expected_answer","license" ,"used_in" ,"user_name" ,"user_url" ,"url"], inplace=True ,axis=1)
#train_data.dropna(inplace=True)
train_data = train_data[train_data["tools"].isna()]
train_data.drop(columns=["tools"], inplace=True ,axis=1)
def remove_none_keys(messages):
    return [{k: v for k, v in entry.items() if v is not None} for entry in messages]
def format_for_gpt_oss(example):
    messages = example['messages']
    new_messages = []
    
    for msg in messages:
        new_msg = msg.copy()
        
        # 1. Rename 'reasoning_content' to 'thinking'
        if 'reasoning_content' in new_msg:
            new_msg['thinking'] = new_msg.pop('reasoning_content')
        
        # 2. Ensure intermediate tool steps don't conflict
        # The template raises an error if you have BOTH 'thinking' and 'content' 
        # inside a tool call message. Your data has content='', which is fine, 
        # but purely safe practice is to ensure it is None or empty.
        if new_msg.get('tool_calls') and new_msg.get('thinking'):
             new_msg['content'] = "" # Ensure this is empty to avoid template error

        new_messages.append(new_msg)
    
    return {'messages': new_messages}

# Apply to your dataset
train_data_formatted = train_data.apply(format_for_gpt_oss, axis=1)
train_data["messages"] = train_data_formatted
train_data["messages"].iloc[0]["messages"]
train_data["AA"] = train_data["messages"].apply(lambda x: x["messages"])
dataset = train_data.iloc[0:cfg.TRAIN_SIZE].copy()
dataset["AA"] = dataset["AA"].apply(remove_none_keys)

dataset["text"] = dataset.apply(lambda row: tokenizer.apply_chat_template(
    row["AA"], 
    tokenize=False, 
    add_generation_prompt=True,
    reasoning_effort="low"
), axis=1)


In [ ]:
dataset["text"].iloc[0]

In [ ]:
# Add LoRA adapters with rank 16
hf_dataset = Dataset.from_pandas(dataset)
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 64,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=hf_dataset,
    dataset_text_field="text",
    max_seq_length=cfg.MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    args=SFTConfig(
        per_device_train_batch_size=cfg.BATCH_SIZE,
        gradient_accumulation_steps=1,
        warmup_steps=5,
        #num_train_epochs=0.25, 
        max_steps=10 ,
        learning_rate=cfg.LEARNING_RATE,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
    ),
)

gpt_oss_kwargs = dict(
    instruction_part="<|start|>user<|message|>", 
    response_part="<|start|>assistant<|channel|>final<|message|>"
)

trainer = train_on_responses_only(
    trainer,
    **gpt_oss_kwargs,
)

In [ ]:
trainer_stats = trainer.train()

used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"Peak reserved memory = {used_memory} GB")

In [ ]:
model.save_pretrained("gpt_oss_20b_nemotronv2_low")
tokenizer.save_pretrained("gpt_oss_20b_nemotronv2_low")
print("Model saved to 'gpt_oss_20b_nemotronv2_low'")